# CO5085 — Chạy toàn bộ bài tập E1–E3

1. **Runtime → Change runtime type → T4 GPU** (hoặc GPU bất kỳ), rồi chạy lần lượt từng ô.
2. Ô đầu tiên lấy mã nguồn: điền `REPO_URL` nếu repo đã lên GitHub, hoặc để trống để tải file `co5085-exercises.zip` lên.
3. Cuối notebook tải về `results_for_report.zip`. Dùng file này để sinh bảng/hình và biên dịch báo cáo.

Thời gian ước tính trên T4: E1 khoảng 3 phút, E2 khoảng 20–30 phút, E3 khoảng 5–10 phút.

In [ ]:
REPO_URL = ""   # ví dụ "https://github.com/TamTran9x/2470743-CO5085-Exercise1.git"; để trống nếu tải zip lên

import os
if not os.path.isdir("co5085-exercises"):
    if REPO_URL:
        !git clone -q "$REPO_URL" co5085-exercises
    else:
        from google.colab import files
        uploaded = files.upload()          # chọn co5085-exercises.zip
        zip_name = next(iter(uploaded))
        !unzip -q -o "$zip_name" -d .
%cd co5085-exercises
!ls

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())

## 0. Kiểm tra tính đúng
Tất cả test phải PASS trước khi huấn luyện.

In [ ]:
!python -m tests.test_all

## 1. E1 — Softmax, MLP, CNN

In [ ]:
!python -m E1.train --model all --epochs 15

## 2. E2 — MSA tự viết vs PyTorch, các cách tokenize
Kiểm chứng MSA tự viết (so với `nn.MultiheadAttention` khi dùng chung trọng số), đo thời gian theo số token, rồi huấn luyện 6 cấu hình.

In [ ]:
!python -m E2.check_msa
!python -m E2.bench_msa

In [ ]:
!python -m E2.train --epochs 20

Tùy chọn: thêm biến thể patch chồng nhau (169 token, chậm hơn khoảng 3–4 lần).

In [ ]:
# !python -m E2.train --runs patch4s2:scratch --epochs 20

## 3. E3 — LSTM / GRU

In [ ]:
!python -m E3.train --epochs 15
!python -m E3.temporal

## 4. Sinh bảng, hình, số liệu cho báo cáo

In [ ]:
!python scripts/make_report_assets.py
!ls reports/generated

## 5. Tải kết quả về
`results_for_report.zip` không chứa checkpoint `.pt` (nhẹ, đủ để dựng báo cáo). `results_full.zip` có cả checkpoint.

In [ ]:
!zip -qr results_for_report.zip results reports/generated -x "*.pt"
!zip -qr results_full.zip results
from google.colab import files
files.download("results_for_report.zip")

## 6. (Tùy chọn) Biên dịch PDF ngay trên Colab
Cài TeX mất vài phút. Nếu lỗi gói, hãy dùng Overleaf (compiler XeLaTeX) với thư mục `reports/`.

In [ ]:
# !apt-get -qq install -y texlive-xetex texlive-latex-extra texlive-lang-other fonts-texgyre fonts-dejavu > /dev/null
# %cd reports
# !xelatex -interaction=nonstopmode exercises.tex > /dev/null; xelatex -interaction=nonstopmode exercises.tex | tail -2
# %cd ..
# files.download("reports/exercises.pdf")